# Notebook 03 — Naive Bayes Baseline

**Purpose**: Train and evaluate Naive Bayes classifiers for:
- Task A: Binary classification (sarcastic vs non-sarcastic)
- Task B: Sarcasm type classification (6-class)

Compares:
- `CountVectorizer + MultinomialNB`
- `TfidfVectorizer + MultinomialNB`
- `CountVectorizer + ComplementNB` (for imbalanced type task)

**Prerequisite**: Run `01_data_preparation.ipynb` first.

**Outputs** in `outputs/classical/naive_bayes/`

In [ ]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Robust project-root finder ─────────────────────────────────────────────
def _find_project_root() -> Path:
    """Walk up from cwd until we find outputs/splits or another project marker."""
    markers = [
        Path("outputs") / "splits",
        Path("outputs") / "datasets",
        Path("data") / "processed",
        Path("notebooks"),
    ]
    for root in [Path.cwd()] + list(Path.cwd().parents):
        for marker in markers:
            if (root / marker).exists():
                return root
    return Path.cwd().parent  # fallback

ROOT    = _find_project_root()
SPLITS  = ROOT / "outputs" / "splits"
OUT_DIR = ROOT / "outputs" / "classical" / "naive_bayes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root    : {ROOT}")
print(f"Output directory: {OUT_DIR}")


## Helper Functions

In [ ]:
def load_splits(task: str):
    train = pd.read_csv(SPLITS / f"train_{task}.csv")
    val   = pd.read_csv(SPLITS / f"val_{task}.csv")
    test  = pd.read_csv(SPLITS / f"test_{task}.csv")
    return train, val, test


def evaluate(model, X, y_true, label_names=None, split_name=""):
    y_pred = model.predict(X)
    metrics = {
        "split"           : split_name,
        "accuracy"        : accuracy_score(y_true, y_pred),
        "precision_macro" : precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro"    : recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro"        : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted"     : f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    print(f"[{split_name}] Accuracy={metrics['accuracy']:.4f}  "
          f"Macro-F1={metrics['f1_macro']:.4f}  Weighted-F1={metrics['f1_weighted']:.4f}")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    return metrics, y_pred


def save_confusion_matrix(y_true, y_pred, label_names, out_path, title=""):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(label_names)))
    fig, ax = plt.subplots(figsize=(max(6, len(label_names)), max(5, len(label_names))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path.relative_to(ROOT)}")


def run_nb_grid(
    vectorizer_cls, nb_cls, param_grid: dict,
    X_trainval, y_trainval, groups_tv,
    name: str
) -> GridSearchCV:
    """Run a grid search for a given vectorizer + NB combination."""
    pipe = Pipeline([
        ("vec", vectorizer_cls(lowercase=True)),
        ("nb",  nb_cls()),
    ])
    gs = GridSearchCV(
        pipe, param_grid, cv=GroupKFold(5),
        scoring="f1_macro", n_jobs=-1, verbose=0, refit=True
    )
    gs.fit(X_trainval, y_trainval, groups=groups_tv)
    print(f"{name:50s}  CV F1={gs.best_score_:.4f}  params={gs.best_params_}")
    return gs

## Task A â€” Binary Classification

In [ ]:
train_bin, val_bin, test_bin = load_splits("binary")
trainval_bin = pd.concat([train_bin, val_bin], ignore_index=True)

X_tv   = trainval_bin["text"].tolist()
y_tv   = trainval_bin["binary_label"].tolist()
grp_tv = trainval_bin["group_id"].tolist()

X_val  = val_bin["text"].tolist();  y_val  = val_bin["binary_label"].tolist()
X_test = test_bin["text"].tolist(); y_test = test_bin["binary_label"].tolist()

label_names_bin = ["non-sarcastic", "sarcastic"]

print(f"Train+Val: {len(X_tv):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

In [ ]:
# â”€â”€ Shared grid parameters â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
BASE_GRID = {
    "vec__ngram_range": [(1, 1), (1, 2)],
    "vec__min_df"     : [2, 3, 5],
    "nb__alpha"       : [0.1, 0.5, 1.0],
}

print("=== Binary: Comparing Vectorizer + NB Combinations ===")

gs_count_mnb_bin = run_nb_grid(
    CountVectorizer, MultinomialNB, BASE_GRID,
    X_tv, y_tv, grp_tv,
    "CountVectorizer + MultinomialNB"
)

gs_tfidf_mnb_bin = run_nb_grid(
    TfidfVectorizer, MultinomialNB, BASE_GRID,
    X_tv, y_tv, grp_tv,
    "TfidfVectorizer + MultinomialNB"
)

gs_count_cnb_bin = run_nb_grid(
    CountVectorizer, ComplementNB, BASE_GRID,
    X_tv, y_tv, grp_tv,
    "CountVectorizer + ComplementNB"
)

In [ ]:
# â”€â”€ Select best binary model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
candidates_bin = [
    ("CountVec+MultinomialNB", gs_count_mnb_bin),
    ("TfidfVec+MultinomialNB", gs_tfidf_mnb_bin),
    ("CountVec+ComplementNB",  gs_count_cnb_bin),
]

best_name_bin, best_gs_bin = max(candidates_bin, key=lambda x: x[1].best_score_)
print(f"\nBest binary NB model: {best_name_bin} (CV F1={best_gs_bin.best_score_:.4f})")

# Comparison table
comp_df = pd.DataFrame([
    {"model": name, "cv_f1_macro": gs.best_score_, "best_params": str(gs.best_params_)}
    for name, gs in candidates_bin
])
print("\nComparison table:")
print(comp_df.to_string(index=False))

In [ ]:
# â”€â”€ Evaluate best model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best_model_bin = best_gs_bin.best_estimator_

val_m_bin,  y_val_pred_bin  = evaluate(best_model_bin, X_val,  y_val,  label_names_bin, "Val")
test_m_bin, y_test_pred_bin = evaluate(best_model_bin, X_test, y_test, label_names_bin, "Test")

# Also evaluate all three models on test for comparison
print("\n--- All models on Test ---")
all_test_results_bin = []
for name, gs in candidates_bin:
    y_pred_t = gs.best_estimator_.predict(X_test)
    f1 = f1_score(y_test, y_pred_t, average="macro")
    acc = accuracy_score(y_test, y_pred_t)
    all_test_results_bin.append({"model": name, "accuracy": acc, "f1_macro": f1})
    print(f"  {name:40s}: acc={acc:.4f}  macro-F1={f1:.4f}")

In [ ]:
# â”€â”€ Save binary results â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
cfg_bin = {
    "task": "binary", "best_model": best_name_bin,
    "best_params": best_gs_bin.best_params_, "cv_f1_macro": best_gs_bin.best_score_,
    "all_candidates": [
        {"model": name, "cv_f1_macro": gs.best_score_, "params": gs.best_params_}
        for name, gs in candidates_bin
    ]
}
with open(OUT_DIR / "best_config_binary.json", "w") as f:
    json.dump(cfg_bin, f, indent=2)

all_m_bin = {"val": val_m_bin, "test": test_m_bin,
             "all_test_comparison": all_test_results_bin}
with open(OUT_DIR / "metrics_binary.json", "w") as f:
    json.dump(all_m_bin, f, indent=2)

save_confusion_matrix(
    y_val, y_val_pred_bin, label_names_bin,
    OUT_DIR / "confusion_matrix_binary_val.png",
    f"NB ({best_name_bin}) â€” Binary (Val)"
)
save_confusion_matrix(
    y_test, y_test_pred_bin, label_names_bin,
    OUT_DIR / "confusion_matrix_binary_test.png",
    f"NB ({best_name_bin}) â€” Binary (Test)"
)

# Predictions
pred_val_bin = val_bin.copy()
pred_val_bin["predicted"] = y_val_pred_bin
pred_val_bin["correct"]   = (pred_val_bin["binary_label"] == pred_val_bin["predicted"]).astype(int)
pred_val_bin.to_csv(OUT_DIR / "predictions_val_binary.csv", index=False)

pred_test_bin = test_bin.copy()
pred_test_bin["predicted"] = y_test_pred_bin
pred_test_bin["correct"]   = (pred_test_bin["binary_label"] == pred_test_bin["predicted"]).astype(int)
pred_test_bin.to_csv(OUT_DIR / "predictions_test_binary.csv", index=False)

print("All binary artifacts saved.")

## Task B â€” Sarcasm Type Classification

In [ ]:
train_type, val_type, test_type = load_splits("type")

STRATEGY_LABELS = sorted(train_type["type_label"].unique())
label2id = {lab: i for i, lab in enumerate(STRATEGY_LABELS)}
id2label = {i: lab for lab, i in label2id.items()}

def enc(df): return [label2id[l] for l in df["type_label"]]

trainval_type = pd.concat([train_type, val_type], ignore_index=True)
X_tv_t   = trainval_type["text"].tolist()
y_tv_t   = enc(trainval_type)
grp_tv_t = trainval_type["group_id"].tolist()

X_val_t  = val_type["text"].tolist();  y_val_t  = enc(val_type)
X_test_t = test_type["text"].tolist(); y_test_t = enc(test_type)

print(f"Strategies: {STRATEGY_LABELS}")
print(f"Train+Val: {len(X_tv_t):,}  Val: {len(X_val_t):,}  Test: {len(X_test_t):,}")

In [ ]:
print("=== Type: Comparing Vectorizer + NB Combinations ===")

gs_count_mnb_type = run_nb_grid(
    CountVectorizer, MultinomialNB, BASE_GRID,
    X_tv_t, y_tv_t, grp_tv_t,
    "CountVectorizer + MultinomialNB"
)

gs_tfidf_mnb_type = run_nb_grid(
    TfidfVectorizer, MultinomialNB, BASE_GRID,
    X_tv_t, y_tv_t, grp_tv_t,
    "TfidfVectorizer + MultinomialNB"
)

gs_count_cnb_type = run_nb_grid(
    CountVectorizer, ComplementNB, BASE_GRID,
    X_tv_t, y_tv_t, grp_tv_t,
    "CountVectorizer + ComplementNB  (for imbalance)"
)

In [ ]:
candidates_type = [
    ("CountVec+MultinomialNB", gs_count_mnb_type),
    ("TfidfVec+MultinomialNB", gs_tfidf_mnb_type),
    ("CountVec+ComplementNB",  gs_count_cnb_type),
]

best_name_type, best_gs_type = max(candidates_type, key=lambda x: x[1].best_score_)
print(f"Best type NB model: {best_name_type} (CV F1={best_gs_type.best_score_:.4f})")

print("\n--- All models on Test (Type Task) ---")
all_test_results_type = []
for name, gs in candidates_type:
    y_pred_t = gs.best_estimator_.predict(X_test_t)
    f1  = f1_score(y_test_t, y_pred_t, average="macro")
    acc = accuracy_score(y_test_t, y_pred_t)
    all_test_results_type.append({"model": name, "accuracy": acc, "f1_macro": f1})
    print(f"  {name:40s}: acc={acc:.4f}  macro-F1={f1:.4f}")

In [ ]:
best_model_type = best_gs_type.best_estimator_

val_m_type,  y_val_pred_type  = evaluate(best_model_type, X_val_t,  y_val_t,  STRATEGY_LABELS, "Val")
test_m_type, y_test_pred_type = evaluate(best_model_type, X_test_t, y_test_t, STRATEGY_LABELS, "Test")

# Save
cfg_type = {
    "task": "type", "best_model": best_name_type,
    "label_names": STRATEGY_LABELS, "label2id": label2id,
    "best_params": best_gs_type.best_params_, "cv_f1_macro": best_gs_type.best_score_,
}
with open(OUT_DIR / "best_config_type.json", "w") as f:
    json.dump(cfg_type, f, indent=2)

all_m_type = {"val": val_m_type, "test": test_m_type,
              "all_test_comparison": all_test_results_type}
with open(OUT_DIR / "metrics_type.json", "w") as f:
    json.dump(all_m_type, f, indent=2)

save_confusion_matrix(
    y_val_t, y_val_pred_type, STRATEGY_LABELS,
    OUT_DIR / "confusion_matrix_type_val.png",
    f"NB ({best_name_type}) â€” Type (Val)"
)
save_confusion_matrix(
    y_test_t, y_test_pred_type, STRATEGY_LABELS,
    OUT_DIR / "confusion_matrix_type_test.png",
    f"NB ({best_name_type}) â€” Type (Test)"
)

# Predictions with string labels
pred_val_type  = val_type.copy()
pred_test_type = test_type.copy()
for df, y_pred, y_true in [(pred_val_type, y_val_pred_type, y_val_t),
                            (pred_test_type, y_test_pred_type, y_test_t)]:
    df["predicted_label"] = [id2label[i] for i in y_pred]
    df["true_label"]      = [id2label[i] for i in y_true]
    df["correct"]         = (df["type_label"] == df["predicted_label"]).astype(int)

pred_val_type.to_csv( OUT_DIR / "predictions_val_type.csv",  index=False)
pred_test_type.to_csv(OUT_DIR / "predictions_test_type.csv", index=False)

print("All type artifacts saved.")

## Error Examples

In [ ]:
# â”€â”€ Binary error examples â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
err_bin = pred_test_bin[pred_test_bin["correct"] == 0].copy()
fp_bin  = err_bin[err_bin["binary_label"] == 0].head(10)  # predicted sarcastic, actually not
fn_bin  = err_bin[err_bin["binary_label"] == 1].head(10)  # predicted not-sarcastic, actually sarcastic

print(f"Binary errors on test: {len(err_bin)} total")
print(f"  False Positives (predicted sarcastic, actually not): {len(err_bin[err_bin['binary_label']==0])}")
print(f"  False Negatives (predicted not-sarcastic, actually sarcastic): {len(err_bin[err_bin['binary_label']==1])}")
print("\n--- Sample FP ---")
for _, row in fp_bin.iterrows():
    print(f"  [True=0, Pred=1] {row['text'][:100]}")
print("\n--- Sample FN ---")
for _, row in fn_bin.iterrows():
    print(f"  [True=1, Pred=0] {row['text'][:100]}")

In [ ]:
# â”€â”€ Type error examples â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
err_type = pred_test_type[pred_test_type["correct"] == 0].copy()
print(f"Type errors on test: {len(err_type)} total")
print("\nConfusion pairs (true â†’ predicted):")
conf_pairs = err_type.groupby(["type_label", "predicted_label"]).size().sort_values(ascending=False).head(10)
print(conf_pairs.to_string())

## Summary

In [ ]:
print("====== NAIVE BAYES RESULTS SUMMARY ======")
print()
print("Task A â€” Binary (Test):")
print(f"  Best model    : {best_name_bin}")
print(f"  Accuracy      : {test_m_bin['accuracy']:.4f}")
print(f"  Macro-F1      : {test_m_bin['f1_macro']:.4f}")
print(f"  Weighted-F1   : {test_m_bin['f1_weighted']:.4f}")
print()
print("Task B â€” Type (Test):")
print(f"  Best model    : {best_name_type}")
print(f"  Accuracy      : {test_m_type['accuracy']:.4f}")
print(f"  Macro-F1      : {test_m_type['f1_macro']:.4f}")
print(f"  Weighted-F1   : {test_m_type['f1_weighted']:.4f}")